In [8]:
# Import Required Libraries
import traceback
import pandas as pd
import numpy as np
import json
import ast
import uuid
import logging
import re
from datetime import date, datetime, timedelta
from math import ceil
from concurrent.futures import ThreadPoolExecutor

import psycopg2
from psycopg2 import sql, DatabaseError
from psycopg2.extras import execute_values

import pymssql

# Setting up logging configuration
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def read_global_etl_batch_id(conn):
    """
    Reads the current running batch information from global batch audit table.
    
    Returns:
        tuple: (batch_run_id, run_end_datetime, load_type) or (None, None, None) if no running batch
        
    Raises:
        Exception: If error occurs while reading batch information
    """
    try:
        read_query = """
            SELECT * FROM bronze_etl_master.bronze_etl_global_batch_audit begba 
            WHERE begba.run_status = 'RUNNING'
            ORDER BY batch_run_id DESC, run_start_datetime DESC
        """
        
        batch_run_id = pd.read_sql_query(read_query, conn)
        
        if batch_run_id.empty:
            logging.info("No running batch found.")
            return None, None, None, None
        else:
            logging.info(f"Running batch found with batch_run_id: {batch_run_id.iloc[0]['batch_run_id']}")
            return (
                batch_run_id.iloc[0]["batch_run_id"], 
                batch_run_id.iloc[0]["run_end_datetime"], 
                batch_run_id.iloc[0]["load_type"],
                batch_run_id.iloc[0]["run_frequency"]
            )
    except Exception as e:
        error_msg = f"Error while reading global ETL batch ID: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def initialize_database_connection(db_type, **kwargs):
    """
    Initializes and returns a database connection dynamically based on the provided database type.
    
    Args:
        db_type (str): Type of database ('postgres', 'mssql', 'mysql')
        **kwargs: Database connection parameters
        
    Returns:
        connection object
        
    Raises:
        Exception: If connection fails
    """
    try:
        if db_type.lower() == "postgres":
            connection = psycopg2.connect(
                dbname=kwargs["pg_dbname"],
                user=kwargs["pg_user"],
                password=kwargs["pg_password"],
                host=kwargs["pg_host"],
                port=kwargs["pg_port"],
            )
            logging.info("PostgreSQL connection established successfully.")

        elif db_type.lower() == "mssql":
            connection = pymssql.connect(
                server=kwargs["mssql_server"],
                user=kwargs["mssql_user"],
                password=kwargs["mssql_pass"],
                database=kwargs["mssql_db"],
                port=kwargs.get("mssql_port", 1433),
            )
            logging.info("MSSQL connection established successfully.")

        elif db_type.lower() == "mysql":
            connection = mysql.connector.connect(
                host=kwargs["host"],
                user=kwargs["user"],
                password=kwargs["password"],
                database=kwargs["database"],
                port=kwargs.get("port", 3306),
            )
            logging.info("MySQL connection established successfully.")

        else:
            raise Exception(f"Unsupported database type: {db_type}")

        return connection

    except Exception as e:
        error_msg = f"Failed to establish connection to {db_type} database. Error: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


import ast
import logging
import pandas as pd

logger = logging.getLogger(__name__)

def read_etl_config_table(
    connection,
    source_name,
    table_type,
    source_db_code,
    source_table_name,
    run_type="DELTA",
    run_frequency='HOURLY',
): 
    """
    Reads ETL configuration table safely and dynamically.
    """

    try:
        def safe_literal_eval(value):
            if isinstance(value, (list, tuple)):
                return tuple(value)
            try:
                return tuple(ast.literal_eval(value))
            except (SyntaxError, ValueError, TypeError):
                return tuple()

        # Normalize source_name
        
        source_name = safe_literal_eval(source_name)
        table_name = safe_literal_eval(source_table_name)
        source_db_code_val = safe_literal_eval(source_db_code)

        # Base Query
        query = """
            SELECT
                ssm.source_system_id,
                sdm.source_db_id,
                sdm.source_db_code,
                ssm.source_system_name,
                ssm.is_multi_db,
                sdm.db_name,
                sdm.db_connection_type,
                sdm.market_profile_id,
                blect.load_sequence,
                blect.source_table_name,
                blect.source_schema_name,
                blect.extract_key,
                blect.load_type,
                blect.table_type,
                blect.target_database_name,
                blect.target_schema_name,
                blect.target_table_name,
                blect.load_key,
                blect.hierarchy_type,
                blect.hierarchy_number,
                blect.is_dependent_on_transaction,
                blect.dependent_tables,
                blect.batch_size,
                blect.run_frequency,
                blect.is_active
            FROM bronze_etl_master.source_system_master ssm
            JOIN bronze_etl_master.source_db_master sdm
                ON ssm.source_system_id = sdm.source_system_id
            JOIN bronze_etl_master.bronze_etl_config_tbl blect
                ON ssm.source_system_id = blect.source_system_id
            WHERE
                ssm.is_active = 1
                AND sdm.is_active = 1
                AND blect.is_active = 1
                AND ssm.source_system_name IN %s
                AND blect.table_type = %s
        """

        params = [source_name, table_type]

        # Optional source_db_code filter
        if source_db_code_val:
            query += " AND sdm.source_db_code IN %s"
            params.append(source_db_code_val)

        # Optional table_name filter
        if table_name:
            query += " AND blect.source_table_name IN %s"
            params.append(table_name)

        # DELTA frequency filter
        if run_type.lower() == "delta":
            query += " AND blect.run_frequency = %s"
            params.append(run_frequency)

        logger.info(
            f"Executing ETL config query | source_system={source_name} | "
            f"run_type={run_type} | run_frequency={run_frequency}"
        )

        logger.info(f"Params: {params}")

        parameter_tbl = pd.read_sql_query(query, connection, params=tuple(params))

        logger.info(f"Rows fetched from ETL config: {len(parameter_tbl)}")

        return parameter_tbl

    except Exception as e:
        error_msg = (
            f"Error while reading ETL config table | "
            f"source_system={source_name} | table_type={table_type}: {str(e)}"
        )
        logger.exception(error_msg)
        raise Exception(error_msg)



def update_audit_master_tbl(
    connection_dl_bronze,
    batch_run_id,
    etl_name,
    etl_start_time=None,
    etl_end_time=None,
    status=None,
    table_processed=0,
    table_succeeded=0,
    table_failed=0,
    error_msg=None,
    process_id=None,
    etl_schedule_time=None
):
    """
    Inserts or updates an ETL process record in the master audit table.
    
    Raises:
        Exception: If audit update fails
    """
    try:
        cursor = connection_dl_bronze.cursor()

        if status and status.lower() == "in-progress":
            if process_id == None:
                cursor.execute("""
                    SELECT COALESCE(MAX(master_audit_id), 0) + 1 AS process_id
                    FROM bronze_etl_master.bronze_etl_run_master_audit_tbl_test
                """)
                process_id = cursor.fetchone()[0]

            insert_query = """
                INSERT INTO bronze_etl_master.bronze_etl_run_master_audit_tbl_test (
                    master_audit_id,
                    etl_batch_id,
                    schedule_name,
                    schedule_timestamp,
                    status,
                    start_time,
                    end_time,
                    tables_processed,
                    tables_successful,
                    tables_failed,
                    error_message
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            cursor.execute(insert_query, (
                int(process_id),
                int(batch_run_id),
                etl_name,
                etl_schedule_time,
                status,
                etl_start_time,
                etl_end_time,
                table_processed,
                table_succeeded,
                table_failed,
                error_msg,
            ))
            logging.info(f"New Process Started with Process ID: {process_id}.")
        else:
            if process_id == None:
                raise Exception("Unable to process without Process ID")
            update_query = """
                UPDATE bronze_etl_master.bronze_etl_run_master_audit_tbl_test
                SET 
                    status = %s,
                    end_time = %s,
                    tables_processed = %s,
                    tables_successful = %s,
                    tables_failed = %s,
                    error_message = %s
                WHERE master_audit_id = %s
            """
            cursor.execute(update_query, (
                status,
                etl_end_time,
                table_processed,
                table_succeeded,
                table_failed,
                error_msg,
                process_id,
            ))
            logging.info(f"Process Completed for Process ID: {process_id}.")
        connection_dl_bronze.commit()
        return process_id
    except Exception as e:
        error_msg = f"Failed to update audit master table: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def upsert_etl_audit_record(
    connection_dl_bronze,
    process_id,
    batch_run_id,
    source_name,
    source_table,
    destination_table,
    operation,
    time_from,
    input_row,
    status,
    db_code='MB-101',
    time_to=None,
    output_row=0,
    error_code=None,
    result=None,
    etl_end_time=None,
    etl_start_time=None
):
    """
    Inserts or updates a record in the ETL audit table.
    
    Raises:
        Exception: If audit log operation fails
    """
    if db_code == None:
        db_code = 'MB-101'
    
    try:
        input_row = int(input_row) if input_row is not None else None
        output_row = int(output_row) if output_row is not None else None

        with connection_dl_bronze.cursor() as cur:
            if status.lower() == "in-progress":
                insert_to_audit_tbl_query = """
                    INSERT INTO bronze_etl_master.bronze_etl_run_table_audit_log (
                        process_id, batch_run_id, data_source, db_code, source_table_name, destination_table_name,
                        operation, time_from, time_to, etl_start_time, etl_end_time,
                        input_row, output_row, operation_result, errorcode, result
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """
                cur.execute(insert_to_audit_tbl_query, (
                    int(process_id),
                    int(batch_run_id),
                    source_name,
                    db_code,
                    source_table,
                    destination_table,
                    operation,
                    time_from,
                    time_to,
                    etl_start_time,
                    time_to,
                    input_row,
                    output_row,
                    status,
                    error_code,
                    result,
                ))
                logging.info(f"ETL Process start logged successfully.")
            else:
                update_audit_tbl_query = """
                    UPDATE bronze_etl_master.bronze_etl_run_table_audit_log
                    SET time_to = %s,
                        etl_end_time = %s,
                        input_row = %s,
                        output_row = %s,
                        operation_result = %s,
                        errorcode = %s,
                        result = %s
                    WHERE process_id = %s AND batch_run_id = %s AND source_table_name = %s
                """
                cur.execute(update_audit_tbl_query, (
                    time_to,
                    time_to,
                    int(input_row),
                    int(output_row),
                    status,
                    error_code,
                    result,
                    int(process_id),
                    int(batch_run_id),
                    source_table,
                ))
                logging.info(f"ETL Process Completion logged successfully.")
            connection_dl_bronze.commit()
    except Exception as e:
        error_msg = f"Audit log failed: {str(e)}"
        logging.error(error_msg)
        connection_dl_bronze.rollback()
        raise Exception(error_msg)


def ensure_uuid_strings_auto(df: pd.DataFrame) -> pd.DataFrame:
    """
    Auto-detect UUID values and convert them to uppercase strings.
    """
    uuid_regex = re.compile(
        r"^[0-9a-fA-F]{8}-"
        r"[0-9a-fA-F]{4}-"
        r"[1-5][0-9a-fA-F]{3}-"
        r"[89abAB][0-9a-fA-F]{3}-"
        r"[0-9a-fA-F]{12}$"
    )

    def convert(val):
        if isinstance(val, uuid.UUID):
            return str(val).upper()
        elif isinstance(val, str):
            val = val.strip()
            if uuid_regex.match(val):
                try:
                    return str(uuid.UUID(val)).upper()
                except Exception:
                    return val
        return val

    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].apply(convert)

    return df


def clean_dataframe_for_db(df: pd.DataFrame, uuid_columns=None) -> pd.DataFrame:
    """
    Cleans the DataFrame for safe database insertion.
    """
    df_cleaned = df.copy()
    for col in df_cleaned.columns:
        if df_cleaned[col].dtype == object or pd.api.types.is_string_dtype(df_cleaned[col]):
            df_cleaned[col] = df_cleaned[col].apply(
                lambda x: (
                    None
                    if isinstance(x, str) and ("\x00" in x or x.strip() == "")
                    else x
                )
            )

    df_cleaned = df_cleaned.where(pd.notnull(df_cleaned), None)
    df_cleaned = ensure_uuid_strings_auto(df_cleaned)
    return df_cleaned


def get_json_and_array_columns(pg_conn, schema_name, table_name):
    """
    Fetches the JSON and array columns for a given PostgreSQL table.
    
    Raises:
        Exception: If query fails
    """
    try:
        json_columns = []
        array_columns = []

        query = """
            SELECT column_name, data_type, udt_name
            FROM information_schema.columns
            WHERE LOWER(table_schema) = LOWER(%s)
            AND LOWER(table_name) = LOWER(%s)
        """

        with pg_conn.cursor() as cursor:
            cursor.execute(query, (schema_name, table_name))
            for column_name, data_type, udt_name in cursor.fetchall():
                if data_type.lower() in ["json", "jsonb"]:
                    json_columns.append(column_name)
                elif udt_name.startswith("_"):
                    array_columns.append(column_name)

        return json_columns, array_columns
    except Exception as e:
        error_msg = f"Failed to get JSON/array columns for {schema_name}.{table_name}: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def write_batch_to_datalake(
    pg_conn,
    batch_df,
    target_schema,
    target_table,
    load_type,
    conflict_columns=None,
    json_columns=None,
    array_columns=None,
):
    """
    Writes a batch of data from a DataFrame to a PostgreSQL table.
    
    Raises:
        Exception: If write operation fails
    """
    try:
        logging.info("Data Insertion Started.")

        batch_df = clean_dataframe_for_db(batch_df)

        if batch_df.empty:
            logging.info("Empty batch, skipping write.")
            return True

        if "insertion_date" not in batch_df.columns:
            batch_df["insertion_date"] = date.today()

        cleaned_df = batch_df.where(pd.notnull(batch_df), None)
        cleaned_df = cleaned_df.replace({pd.NaT: None, pd.NA: None, float("nan"): None})

        json_columns = json_columns or []
        array_columns = array_columns or []

        def serialize_cell(val, is_json=False, is_array=False):
            try:
                if is_json:
                    return json.dumps(val, default=str)
                if is_array:
                    return f'{{{",".join(map(str, val))}}}' if isinstance(val, list) else None
            except Exception as e:
                logging.warning(f"Serialization failed for value: {val} -> {e}")
                return None
            return val

        for col in cleaned_df.columns:
            if col in json_columns:
                cleaned_df[col] = cleaned_df[col].apply(lambda x: serialize_cell(x, is_json=True))
            elif col in array_columns:
                cleaned_df[col] = cleaned_df[col].apply(lambda x: serialize_cell(x, is_array=True))

        columns = list(cleaned_df.columns)
        values = [tuple(row) for row in cleaned_df.to_numpy()]
        columns_quoted = ", ".join(f'"{col}"' for col in columns)

        if load_type.lower() in ["append", "full"]:
            insert_query = f"""
                INSERT INTO "{target_schema}"."{target_table}" ({columns_quoted})
                VALUES %s
            """
        elif load_type.lower() == "incremental":
            logging.info(f"Conflicting Columns: {conflict_columns}")
            if not conflict_columns or not isinstance(conflict_columns, list):
                raise Exception("'conflict_columns' must be provided for incremental loads.")
            conflict_cols_str = ", ".join(f'"{col}"' for col in conflict_columns)
            update_set = ", ".join(
                f'"{col}" = EXCLUDED."{col}"'
                for col in columns
                if col not in conflict_columns
            )
            insert_query = f"""
                INSERT INTO "{target_schema}"."{target_table}" ({columns_quoted})
                VALUES %s
                ON CONFLICT ({conflict_cols_str})
                DO UPDATE SET {update_set}
            """
        else:
            raise Exception(f"Unknown load_type: {load_type}")

        with pg_conn.cursor() as cursor:
            execute_values(cursor, insert_query, values)
            pg_conn.commit()
            logging.info(f"""Inserted {len(values)} rows into "{target_schema}"."{target_table}".""")
            return True
            
    except Exception as e:
        pg_conn.rollback()
        error_msg = f"Error during batch insert to {target_schema}.{target_table}: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def build_incremental_where_clause(
    extract_key_str, last_processed_date, upper_bound_time=None, dbtype=None
):
    """
    Builds a SQL WHERE clause for incremental extraction.
    
    For DELTA loads:
    - lower_bound: last_processed_date (already has 5-min buffer)
    - upper_bound: upper_bound_time from global_batch_audit
    - WHERE (col > lower_bound AND col < upper_bound)
    
    For FULL loads:
    - upper_bound: upper_bound_time from global_batch_audit
    - WHERE (col < upper_bound)
    """
    if not extract_key_str:
        return ""

    columns = [col.strip() for col in extract_key_str.split(",") if col.strip()]
    if not columns:
        return ""

    def format_date_for_db(date_value, db_type):
        if not date_value:
            return None
        
        if isinstance(date_value, str):
            from datetime import datetime
            try:
                if ' ' in date_value and ':' in date_value:
                    if '.' in date_value:
                        date_value = datetime.strptime(date_value, '%Y-%m-%d %H:%M:%S.%f')
                    else:
                        date_value = datetime.strptime(date_value, '%Y-%m-%d %H:%M:%S')
                else:
                    date_value = datetime.strptime(date_value, '%Y-%m-%d')
            except ValueError:
                pass
        
        if hasattr(date_value, 'strftime'):
            if hasattr(date_value, 'microsecond') and date_value.microsecond:
                date_string = date_value.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
            else:
                date_string = date_value.strftime('%Y-%m-%d %H:%M:%S')
        else:
            date_string = str(date_value)
        
        return f"'{date_string}'"

    conditions = []
    
    # For DELTA: both lower and upper bounds
    if last_processed_date:
        date_lower = format_date_for_db(last_processed_date, dbtype)
        date_upper = format_date_for_db(upper_bound_time, dbtype) if upper_bound_time else None
        
        for col in columns:
            if date_upper:
                condition = f'("{col}" > {date_lower} AND "{col}" < {date_upper})'
            else:
                condition = f'("{col}" > {date_lower})'
            conditions.append(condition)
    # For FULL: only upper bound
    elif upper_bound_time:
        date_upper = format_date_for_db(upper_bound_time, dbtype)
        
        for col in columns:
            condition = f'("{col}" < {date_upper})'
            conditions.append(condition)
    else:
        return ""

    where_clause = "WHERE " + " OR ".join(conditions)
    return where_clause


def extract_source_table_data(
    df_row, connection, last_processed_date=None, batch_size=100000, where_clause=None
):
    """
    Extracts data from a source table in batches.
    
    Raises:
        Exception: If extraction fails
    """
    try:
        source_schema = df_row.get("source_schema_name", "").strip()
        source_table = df_row.get("source_table_name", "").strip()
        db_type = df_row.get("db_connection_type", "").strip().lower()
        extract_key = (df_row.get("extract_key", "") or "").strip()
        load_key = (df_row.get("load_key", "") or "").strip()
        operation = df_row.get("load_type", "").strip()

        if not source_table:
            raise Exception("Source table name is missing in the parameter row.")

        table_name_for_query = f"[{source_table}]" if source_table.lower() == "user" and db_type == "mssql" else source_table
        table_reference = f'''{source_schema}.{table_name_for_query}''' if source_schema else table_name_for_query

        order_columns = [keys for keys in (load_key or extract_key).split(",") if keys not in ('source_db_id','market_profile_id')]
        logging.info(f"Order Columns: {order_columns}")
        if not order_columns:
            raise Exception("Both load_key and extract_key are missing; cannot order data for batching.")

        where_clause_str = ""
        if where_clause:
            where_clause_str = where_clause.strip()
            if not where_clause_str.lower().startswith("where"):
                where_clause_str = f"WHERE {where_clause_str}"

        logging.info(f"Extracting Data from: {table_reference}")
        logging.info(f"Ordering by: {order_columns}")
        logging.info(f"Operation type: {operation}")
        logging.info(f"WHERE clause: {where_clause_str if where_clause_str else 'None'}")

        tables_requiring_casting = {"Organization", "PrivateFare", "TravelerProfile"}
        use_custom_select = source_table in tables_requiring_casting

        select_columns = "*"
        end_date_expr = ""
        if use_custom_select:
            if db_type == "postgres":
                column_query = f"""
                    SELECT column_name FROM information_schema.columns
                    WHERE table_schema = %s AND table_name = %s AND column_name != 'EndDate'
                """
                with connection.cursor(cursor_factory=psycopg2.extras.DictCursor) as cursor:
                    cursor.execute(column_query, (source_schema, source_table))
                    columns = [row["column_name"] for row in cursor.fetchall()]
            elif db_type == "mssql":
                column_query = f"""
                    SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
                    WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s AND COLUMN_NAME != 'EndDate'
                """
                with connection.cursor(as_dict=True) as cursor:
                    cursor.execute(column_query, (source_schema, source_table))
                    columns = [row["COLUMN_NAME"] for row in cursor.fetchall()]
            elif db_type == "mysql":
                column_query = f"""
                    SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
                    WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s AND COLUMN_NAME != 'EndDate'
                """
                with connection.cursor(dictionary=True) as cursor:
                    cursor.execute(column_query, (source_schema, source_table))
                    columns = [row["COLUMN_NAME"] for row in cursor.fetchall()]
            else:
                raise Exception(f"Unsupported DB type: {db_type}")

            select_columns = ", ".join(columns)
            if db_type == "mssql":
                end_date_expr = (
                    "CASE WHEN EndDate >= '9999-12-31 23:59:59.000' "
                    "THEN CAST('9999-12-31 22:59:59.000' AS DATETIME2) ELSE EndDate END AS EndDate"
                )
            elif db_type == "postgres":
                end_date_expr = (
                    "CASE WHEN \"EndDate\" >= '9999-12-31 23:59:59'::timestamp "
                    "THEN '9999-12-31 22:59:59'::timestamp ELSE \"EndDate\" END AS \"EndDate\""
                )
            else:
                end_date_expr = (
                    "CASE WHEN EndDate >= '9999-12-31 23:59:59' "
                    "THEN CAST('9999-12-31 22:59:59' AS DATETIME) ELSE EndDate END AS EndDate"
                )
            select_columns = f"{select_columns}, {end_date_expr}"

        offset = 0
        batch_count = 0

        while True:
            if db_type == "postgres":
                order_by = ", ".join([f'"{col}"' for col in order_columns])
            elif db_type == "mysql":
                order_by = ", ".join([f'`{col}`' for col in order_columns])
            else:
                order_by = ", ".join(order_columns)

            if use_custom_select:
                select_part = select_columns
            else:
                select_part = "*"

            if db_type == "postgres":
                query = (
                    f"SELECT {select_part} FROM {table_reference} "
                    f"{where_clause_str} ORDER BY {order_by} LIMIT {batch_size} OFFSET {offset}"
                )
            elif db_type == "mysql":
                query = (
                    f"SELECT {select_part} FROM {table_reference} "
                    f"{where_clause_str} ORDER BY {order_by} LIMIT {batch_size} OFFSET {offset}"
                )
            elif db_type == "mssql":
                query = (
                    f'''SELECT {select_part} FROM {table_reference} '''
                    f"{where_clause_str} ORDER BY {order_by} "
                    f"OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY"
                )
            else:
                raise Exception(f"Unsupported DB type: {db_type}")

            logging.info(f"Executing query: {query}")

            df_chunk = pd.read_sql_query(query, connection)
            if df_chunk.empty:
                if batch_count == 0:
                    logging.warning("No data found in source table!")
                else:
                    logging.info("All batches processed. No more data.")
                break

            batch_count += 1
            yield df_chunk
            logging.info(f"Batch Processed: {batch_count}")
            offset += len(df_chunk)
            if len(df_chunk) < batch_size:
                break
                
    except Exception as e:
        error_msg = f"Failed to extract data from {source_schema}.{source_table}: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def get_last_processed_date(connection, target_table, db_code, upper_bound_time):
    """
    Fetches the maximum date from the audit log for the given table.
    Subtracts 5 minute buffer to ensure no data is missed.
    
    Returns:
        datetime or None: Last processed date with 5-min buffer
        
    Raises:
        Exception: If query fails
    """
    try:
        extract_query = f"""
            SELECT (MAX(time_to) - INTERVAL '5 minutes') as last_processed_date 
            FROM bronze_etl_master.bronze_etl_run_table_audit_log btl
            JOIN bronze_etl_master.bronze_etl_global_batch_audit bgtl on bgtl.batch_run_id = btl.batch_run_id
            WHERE btl.source_table_name = '{target_table}' 
            AND bgtl.load_type = 'DELTA'
            AND btl.etl_start_time IS NOT NULL 
            AND btl.operation_result IN ('Success', 'SUCCESS', 'success')
        """
        
        if db_code:
            extract_query += f"AND btl.db_code = '{db_code}'"
#         logging.info(f"last processed date extract_query: {extract_query}")
        last_processed_date = pd.read_sql_query(extract_query, con=connection)
        if not last_processed_date.empty and 'last_processed_date' in last_processed_date.columns:
            result = last_processed_date.iloc[0]['last_processed_date']
            if pd.notnull(result):
                logging.info(f"Last processed date found: {result} (with 5-min buffer)")
                if result > upper_bound_time:
                    raise Exception(f'''Invalid time frame. lower time bound can't be greater than upper time bound.''')
                else:
                    return result
        
        logging.info("No previous successful run found for this table")
        return None
        
    except Exception as e:
        error_msg = f"Failed to get last processed date for {target_table}: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def process_table(
    connection_source_db, 
    connection_raw_db, 
    row, 
    batch_size, 
    process_id,
    batch_run_id,
    run_type,
    last_processed_date_val=None, 
    dbtype=None, 
    upper_bound_time=None, 
    is_first_db_for_multi=False
):
    """
    Process a table from source to target with various transformations.
    
    Handles both FULL and DELTA loads:
    - FULL: Truncates and loads all data up to upper_bound_time
    - DELTA: Loads incremental data based on operation type:
        * Incremental: Upserts data between lower and upper bounds
        * Append: Deletes then inserts data between bounds
        * Full: Truncates and inserts (respects run_frequency)
    
    Returns:
        tuple: (total_fetched, total_processed, final_status, error_message)
        
    Raises:
        Exception: If processing fails
    """
    final_status = 'Failed'
    total_fetched = 0
    total_processed = 0
    error_message = None
    
    source_schema_name = row['source_schema_name']
    source_table_name = row['source_table_name']
    load_type = row['load_type']
    target_schema_name = row['target_schema_name']
    target_table_name = row['target_table_name']
    extract_key = row['extract_key']
    load_key = row['load_key']
    table_reference = f'''"{source_schema_name}"."{source_table_name}"'''
    source_name = row['source_system_name']
    is_multi_db_source = row['is_multi_db']
    source_db_id = row['source_db_id']
    source_market_profile_id = row['market_profile_id']
    
    try:
        logging.info(f"\n{'='*60}")
        logging.info(f"PROCESSING TABLE: {table_reference}")
        logging.info(f"Run Type: {run_type}")
        logging.info(f"Load Type: {load_type}")
        logging.info(f"Is Multi-DB: {is_multi_db_source}")
        logging.info(f"Is First DB: {is_first_db_for_multi}")
        logging.info(f"{'='*60}")
        
        # Generate Where Clause based on run_type
        where_clause = ""
        
        if run_type.upper() == "FULL":
            # FULL load: process all data up to upper_bound_time
            logging.info("FULL LOAD: Processing all data up to upper_bound_time")
            where_clause = build_incremental_where_clause(
                extract_key_str=extract_key,
                last_processed_date=None,  # No lower bound for FULL
                upper_bound_time=upper_bound_time,
                dbtype=dbtype
            )
            
        elif run_type.upper() == "DELTA":
            # DELTA load: process based on operation type
            logging.info("DELTA LOAD: Processing incremental data")
            
            if load_type.lower() in ("incremental", "append"):
                # For incremental/append: use lower and upper bounds
                last_processed_date_str = str(last_processed_date_val) if last_processed_date_val else None
                
                if last_processed_date_str:
                    logging.info(f"Lower Bound (with 5-min buffer): {last_processed_date_str}")
                    logging.info(f"Upper Bound: {upper_bound_time}")
                    where_clause = build_incremental_where_clause(
                        extract_key_str=extract_key,
                        last_processed_date=last_processed_date_str,
                        upper_bound_time=upper_bound_time,
                        dbtype=dbtype
                    )
                else:
                    # No previous run, process all data up to upper_bound
                    logging.info("No previous run found, processing all data up to upper_bound_time")
                    where_clause = build_incremental_where_clause(
                        extract_key_str=extract_key,
                        last_processed_date=None,
                        upper_bound_time=upper_bound_time,
                        dbtype=dbtype
                    )
                    
            elif load_type.lower() == "full":
                # For full operation in DELTA: truncate and load up to upper_bound
                logging.info("FULL operation in DELTA: Processing all data up to upper_bound_time")
                where_clause = build_incremental_where_clause(
                    extract_key_str=extract_key,
                    last_processed_date=None,
                    upper_bound_time=upper_bound_time,
                    dbtype=dbtype
                )
        else:
            raise Exception(f"Unknown run_type: {run_type}")
        
        logging.info(f"Generated WHERE clause: {where_clause if where_clause else 'None'}")
        
        # Handle table truncation/deletion based on run_type and load_type
        if run_type.upper() == "FULL" or (run_type.upper() == "DELTA" and load_type.lower() == "full"):
            # Truncation logic for FULL loads
            if is_multi_db_source:
                if is_first_db_for_multi:
                    truncate_query = f"""
                        TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY;
                    """
                    with connection_raw_db.cursor() as cursor:
                        cursor.execute(truncate_query)
                        connection_raw_db.commit()
                        logging.info(f"TRUNCATED: {target_schema_name}.{target_table_name} (first DB of multi-db source)")
                else:
                    logging.info(f"SKIPPED TRUNCATION: {target_schema_name}.{target_table_name} (appending from DB {source_db_id})")
            else:
                truncate_query = f"""
                    TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY;
                """
                with connection_raw_db.cursor() as cursor:
                    cursor.execute(truncate_query)
                    connection_raw_db.commit()
                    logging.info(f"TRUNCATED: {target_schema_name}.{target_table_name} (single DB source)")
        
        elif run_type.upper() == "DELTA" and load_type.lower() == 'append' and last_processed_date_val:
            # Append operation: delete data in range, then insert
            delete_query = f"""
                DELETE FROM "{target_schema_name}"."{target_table_name}"
                {where_clause};
            """
            logging.info(f"APPEND operation: Deleting data in range")
            logging.info(f"Delete query: {delete_query}")
            with connection_raw_db.cursor() as cursor:
                cursor.execute(delete_query)
                rows_deleted = cursor.rowcount
                connection_raw_db.commit()
                logging.info(f"DELETED: {rows_deleted} rows from {target_schema_name}.{target_table_name}")
        
        elif run_type.upper() == "DELTA" and load_type.lower() == 'append' and not last_processed_date_val:
            # No previous run found for append ' treat as full load: truncate first
            logging.info(
                "APPEND operation with no previous run found: "
                "falling back to FULL load behaviour (TRUNCATE + INSERT)"
            )
            if is_multi_db_source:
                if is_first_db_for_multi:
                    truncate_query = f"""
                        TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY;
                    """
                    with connection_raw_db.cursor() as cursor:
                        cursor.execute(truncate_query)
                        connection_raw_db.commit()
                        logging.info(
                            f"TRUNCATED: {target_schema_name}.{target_table_name} "
                            f"(first DB of multi-db source, append fallback)"
                        )
                else:
                    logging.info(
                        f"SKIPPED TRUNCATION: {target_schema_name}.{target_table_name} "
                        f"(appending from DB {source_db_id}, append fallback)"
                    )
            else:
                truncate_query = f"""
                    TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY;
                """
                with connection_raw_db.cursor() as cursor:
                    cursor.execute(truncate_query)
                    connection_raw_db.commit()
                    logging.info(
                        f"TRUNCATED: {target_schema_name}.{target_table_name} "
                        f"(single DB source, append fallback)"
                    )
        # Get JSON and Array columns
        conflict_columns = [col.strip() for col in load_key.split(',')] if load_key else []        
        json_cols, array_cols = get_json_and_array_columns(connection_raw_db, target_schema_name, target_table_name)
        batch_count = 0       
        
        # Determine effective load type for writing
        effective_load_type = load_type.lower()
        
        # For multi-db sources after first DB, use append for FULL operations
        if is_multi_db_source and not is_first_db_for_multi and (run_type.upper() == "FULL" or effective_load_type == 'full'):
            effective_load_type = 'append'
            logging.info("Multi-DB source (not first): Using APPEND for write operation")
        
        # Extract and load data in batches
        for batch in extract_source_table_data(row, connection_source_db, last_processed_date=last_processed_date_val, where_clause=where_clause, batch_size=batch_size): 
            batch_count += 1
            input_data_count = len(batch)
            
            logging.info(f"Processing batch {batch_count} with {input_data_count} records")

            total_fetched += input_data_count
            batch['insertion_date'] = datetime.now()
            
            # Add multi-db identifiers
            if is_multi_db_source:
                batch['source_db_id'] = source_db_id
                batch['market_profile_id'] = source_market_profile_id

            # Write batch to datalake
            is_writing_success = write_batch_to_datalake(
                pg_conn=connection_raw_db,
                batch_df=batch,
                target_schema=target_schema_name,
                target_table=target_table_name,
                json_columns=json_cols,
                array_columns=array_cols,
                load_type=effective_load_type,
                conflict_columns=conflict_columns
            )

            if is_writing_success:
                total_processed += len(batch)
                logging.info(f"Batch {batch_count} written successfully")
            else:
                raise Exception(f"Failed to write batch {batch_count} to target table")

            logging.info(f"Progress: Batch {batch_count} | Total Processed: {total_processed:,}")

        logging.info(f"Table processing complete: {batch_count} batch(es) processed")
        
        # Determine final status
        if total_fetched == total_processed and total_fetched > 0:
            final_status = 'Success'
            error_message = None
            logging.info(f"SUCCESS: Fetched={total_fetched:,}, Processed={total_processed:,}")
        elif total_processed > 0:
            final_status = 'Partial Success'
            error_message = f"Processed {total_processed} out of {total_fetched} records"
            logging.warning(f"PARTIAL SUCCESS: {error_message}")
        elif total_fetched == 0 and total_processed == 0:
            final_status = 'Success'
            logging.info("SUCCESS: No records to process (empty result set)")
        else:
            final_status = 'Failed'
            error_message = f"Row count mismatch: Fetched={total_fetched}, Processed={total_processed}"
            logging.error(f"FAILED: {error_message}")
            
    except Exception as e:
        error_message = str(e)
        final_status = 'Failed'
        logging.error(f"FAILED: Table processing error: {error_message}")
        logging.error(f"Traceback: {traceback.format_exc()}")
        raise Exception(f"Error processing table {table_reference}: {error_message}")
        
    return total_fetched, total_processed, final_status, error_message


# ============================================================================
# MAIN ETL FUNCTION
# ============================================================================

def main(
    dl_connection_details,
    sinlge_db_source_host,
    sinlge_db_source_port,
    sinlge_db_source_user,
    sinlge_db_source_password,
    source_name,
    table_type,
    table_name=None,
    source_db_code=None,
    etl_name="BRONZE_ETL",
    multi_db_source_host=None,
    multi_db_source_port=None,
    multi_db_source_user=None,
    multi_db_source_password=None,
):
    """
    Main ETL function that handles both FULL and DELTA loads with multi-db support.
    
    FULL LOAD:
    - Reads all data up to upper_bound_time from global_batch_audit
    - Truncates target table before loading
    - For multi-db sources: only first DB truncates, others append
    
    DELTA LOAD:
    - Incremental: Upserts data between last_processed_date and upper_bound_time
    - Append: Deletes data in range, then inserts fresh data
    - Full: Truncates and loads (respects run_frequency)
    - Uses 5-minute buffer on last_processed_date
    
    Parameters:
        dl_connection_details (dict): Datalake connection credentials
        source_host (str): Source database host
        source_port (str): Source database port
        source_user (str): Source database username
        source_password (str): Source database password
        source_name (str/tuple): Source system name(s)
        table_type (str): 'Master' or 'Transaction'
        source_db_code (str/tuple, optional): Specific DB codes to process
        table_name (str/tuple, optional): Specific tables to process
        etl_name (str): ETL process name for audit
    
    Returns:
        dict: ETL execution results
        
    Raises:
        Exception: If critical error occurs
    """
    process_id = None
    etl_start_time = datetime.now()
    etl_end_time = None
    total_tables = 0
    successful_tables = 0
    failed_tables = 0
    connection_raw_db = None
    overall_error_message = None

    logging.info("=" * 80)
    logging.info(f"ETL STARTED: {etl_name}")
    logging.info(f"Start Time: {etl_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logging.info("=" * 80)

    try:
        # Step 1: Establish datalake connection
        logging.info("Step 1: Establishing datalake connection...")
        connection_raw_db = initialize_database_connection(
            db_type="postgres", **{**dl_connection_details["pg_dl_connection"]}
        )
        logging.info("Datalake connection established")

        # Step 2: Read global batch audit
        logging.info("\nStep 2: Reading global batch audit...")
        batch_run_id, upper_bound_time, run_type, run_frequency = read_global_etl_batch_id(connection_raw_db)
        
        if batch_run_id is None:
            raise Exception("No running batch found in global audit table")
        
        logging.info(f"Batch Run ID: {batch_run_id}")
        logging.info(f"Run Type: {run_type}")
        logging.info(f"Upper Bound Time: {upper_bound_time}")
        logging.info(f"Run Frequency: {run_frequency}")

        # Step 3: Read ETL configuration
        logging.info("\nStep 3: Reading ETL configuration...")
        # run_frequency = None
        
        etl_config_table = read_etl_config_table(
            connection_raw_db,
            source_name,
            table_type,
            source_db_code=source_db_code,
            source_table_name=table_name,
            run_type=run_type,
            run_frequency=run_frequency,
        )
        
        if etl_config_table.empty:
            raise Exception("No tables found matching the criteria")
        
        logging.info(f"Found {len(etl_config_table)} table(s) to process")
        
        # Step 4: Divide into source-specific dataframes
        logging.info("\nStep 4: Dividing configuration by source system...")
        source_systems = etl_config_table['source_system_name'].unique().tolist()
        logging.info(f"Source systems: {source_systems}")
        
        source_results = []
        
        # Step 5: Process each source system
        for source_index, source_system in enumerate(source_systems):
            logging.info(f"\n{'='*80}")
            logging.info(f"PROCESSING SOURCE SYSTEM [{source_index+1}/{len(source_systems)}]: {source_system}")
            logging.info(f"{'='*80}")
            
            source_config = etl_config_table[etl_config_table['source_system_name'] == source_system]
            is_multi_db = source_config.iloc[0]['is_multi_db']
            
            logging.info(f"Is Multi-DB Source: {bool(is_multi_db)}")
            
            if is_multi_db == 0:
                # ============================================================
                # SINGLE DB SOURCE PROCESSING
                # ============================================================
                logging.info("\n--- SINGLE DB SOURCE PROCESSING ---")
                
                db_list = source_config['db_name'].unique().tolist()
                db = db_list[0]
                
                db_config = source_config.iloc[0]
                dbtype = db_config['db_connection_type'].lower()
                db_code = db_config['source_db_code']
                
                logging.info(f"Database: {db}")
                logging.info(f"DB Type: {dbtype}")
                logging.info(f"DB Code: {db_code}")
                
                # Establish source connection
                if dbtype == 'mssql':
                    connection_source = initialize_database_connection(
                        "mssql",
                        mssql_server=sinlge_db_source_host,
                        mssql_user=sinlge_db_source_user,
                        mssql_pass=sinlge_db_source_password,
                        mssql_db=db,
                        mssql_port=sinlge_db_source_port
                    )
                elif dbtype == 'postgres':
                    connection_source = initialize_database_connection(
                        "postgres",
                        pg_dbname=db,
                        pg_user=sinlge_db_source_user,
                        pg_password=sinlge_db_source_password,
                        pg_host=sinlge_db_source_host,
                        pg_port=sinlge_db_source_port
                    )
                elif dbtype == 'mysql':
                    connection_source = initialize_database_connection(
                        "mysql",
                        host=sinlge_db_source_host,
                        user=sinlge_db_source_user,
                        password=sinlge_db_source_password,
                        database=db,
                        port=sinlge_db_source_port
                    )
                else:
                    raise Exception(f"Unsupported database type: {dbtype}")
                
                logging.info(f"Connected to source database: {db}")
                
                # Generate process_id
                source_etl_name = f"{etl_name}_{source_system}"
                source_process_id = update_audit_master_tbl(
                    connection_raw_db,
                    batch_run_id=batch_run_id,
                    etl_name=source_etl_name,
                    etl_start_time=etl_start_time,
                    etl_end_time=None,
                    status="In-Progress",
                    table_processed=0,
                    table_succeeded=0,
                    table_failed=0,
                    error_msg=None,
                    process_id=None,
                    etl_schedule_time=etl_start_time
                )
                logging.info(f"Process ID created: {source_process_id}")
                
                # Sort tables
                source_etl_metainfo = source_config.sort_values(
                    by=["load_sequence", "hierarchy_number"]
                ).reset_index(drop=True)
                
                source_total_tables = 0
                source_successful_tables = 0
                source_failed_tables = 0
                
                # Process each table
                for table_index, row in source_etl_metainfo.iterrows():
                    source_total_tables += 1
                    total_tables += 1
                    
                    table_process_start_time = datetime.now()
                    source_schema = row["source_schema_name"]
                    source_table = row["source_table_name"]
                    destination_table = row["target_table_name"]
                    operation = row["load_type"]
                    
                    logging.info(f"\n{'-'*60}")
                    logging.info(f"Table [{table_index+1}/{len(source_etl_metainfo)}]: {source_schema}.{source_table}")
                    logging.info(f"Target: {row['target_schema_name']}.{destination_table}")
                    logging.info(f"Operation: {operation}")
                    logging.info(f"{'-'*60}")
                    
                    try:
                        # Upsert table audit (in-progress)
                        upsert_etl_audit_record(
                            connection_raw_db,
                            source_process_id,
                            batch_run_id,
                            source_system,
                            source_table,
                            destination_table,
                            operation,
                            db_code=db_code,
                            time_from=table_process_start_time,
                            input_row=0,
                            status="In-Progress",
                            time_to=None,
                            output_row=0,
                            error_code=None,
                            result=None,
                            etl_end_time=None,
                            etl_start_time=etl_start_time
                        )
                        
                        # Get last processed date for DELTA loads
                        last_processed_date_val = None
                        if run_type.upper() == "DELTA" and operation.lower() in ("incremental", "append"):
                            last_processed_date_val = get_last_processed_date(
                                connection_raw_db, 
                                source_table, 
                                db_code,
                                upper_bound_time
                            )  
#                             last_processed_date_val = '2026-01-01 00:00:00.000' # --------------------------Debugging
                        elif run_type.upper() == "DELTA" and operation.lower() == "full":
                            logging.info("DELTA run with FULL operation: processing all data up to upper_bound_time")
                            truncate_query = f"""
                                TRUNCATE TABLE "{row['target_schema_name']}"."{destination_table}" RESTART IDENTITY;
                                """
                            with connection_raw_db.cursor() as cursor:
                                cursor.execute(truncate_query)
                                connection_raw_db.commit()
                                logging.info(f"TRUNCATED: {row['target_schema_name']}.{destination_table} (DELTA run with FULL operation)")
                        
                        batch_size = int(row['batch_size']) if pd.notnull(row['batch_size']) else 100000
                        
                        # Process table
                        total_fetched, total_processed, final_status, error_message = process_table(
                            connection_source_db=connection_source,
                            connection_raw_db=connection_raw_db,
                            row=row,
                            batch_size=batch_size,
                            process_id=source_process_id,
                            batch_run_id=batch_run_id,
                            run_type=run_type,
                            last_processed_date_val=last_processed_date_val,
                            upper_bound_time=upper_bound_time,
                            dbtype=dbtype,
                            is_first_db_for_multi=False
                        )
                        
                        table_end_time = datetime.now()
                        
                        if final_status.lower() in ['success', 'partial success']:
                            source_successful_tables += 1
                            successful_tables += 1
                            result_status = "Successful"
                            logging.info(f"SUCCESS: {source_table} | Fetched: {total_fetched:,} | Processed: {total_processed:,}")
                        else:
                            source_failed_tables += 1
                            failed_tables += 1
                            result_status = "Unsuccessful"
                            logging.error(f"FAILED: {source_table} | Error: {error_message}")
                        
                        # Update table audit
                        upsert_etl_audit_record(
                            connection_raw_db,
                            source_process_id,
                            batch_run_id,
                            source_system,
                            source_table,
                            destination_table,
                            operation,
                            db_code=db_code,
                            time_from=None,
                            input_row=total_fetched,
                            status=final_status,
                            time_to=table_end_time,
                            output_row=total_processed,
                            error_code=error_message,
                            result=result_status,
                        )
                        
                    except Exception as table_error:
                        source_failed_tables += 1
                        failed_tables += 1
                        table_error_message = str(table_error)
                        table_end_time = datetime.now()
                        
                        logging.error(f"TABLE ERROR: {source_table}")
                        logging.error(f"Error: {table_error_message}")
                        logging.error(f"Traceback: {traceback.format_exc()}")
                        
                        try:
                            upsert_etl_audit_record(
                                connection_raw_db,
                                source_process_id,
                                batch_run_id,
                                source_system,
                                source_table,
                                destination_table,
                                operation,
                                db_code=db_code,
                                time_from=None,
                                input_row=0,
                                status="Failed",
                                time_to=table_end_time,
                                output_row=0,
                                error_code=table_error_message,
                                result="Unsuccessful",
                            )
                        except Exception as audit_error:
                            logging.error(f"Audit update failed: {str(audit_error)}")
                
                # Close source connection
                connection_source.close()
                logging.info(f"Closed source connection: {db}")
                
                # Update master audit
                source_etl_end_time = datetime.now()
                
                if source_failed_tables == 0:
                    source_overall_status = "Success"
                    source_overall_error_message = None
                elif source_successful_tables > 0:
                    source_overall_status = "Partial Success"
                    source_overall_error_message = f"{source_failed_tables}/{source_total_tables} tables failed"
                else:
                    source_overall_status = "Failed"
                    source_overall_error_message = "All tables failed"
                
                update_audit_master_tbl(
                    connection_raw_db,
                    batch_run_id=batch_run_id,
                    etl_name=source_etl_name,
                    etl_start_time=None,
                    etl_end_time=source_etl_end_time,
                    status=source_overall_status,
                    table_processed=source_total_tables,
                    table_succeeded=source_successful_tables,
                    table_failed=source_failed_tables,
                    error_msg=source_overall_error_message,
                    process_id=source_process_id,
                )
                
                source_results.append({
                    'source_system': source_system,
                    'is_multi_db': False,
                    'status': source_overall_status,
                    'tables_processed': source_total_tables,
                    'tables_successful': source_successful_tables,
                    'tables_failed': source_failed_tables
                })
                
            else:
                # ============================================================
                # MULTI-DB SOURCE PROCESSING
                # ============================================================
                logging.info("\n--- MULTI-DB SOURCE PROCESSING ---")
                
                db_list = source_config['db_name'].unique().tolist()
                logging.info(f"Databases: {db_list}")
                
                multi_db_truncation_status = {}
                
                source_total_tables = 0
                source_successful_tables = 0
                source_failed_tables = 0
                
                # Process each database
                for db_index, db in enumerate(db_list):
                    logging.info(f"\n{'-'*70}")
                    logging.info(f"Database [{db_index+1}/{len(db_list)}]: {db}")
                    logging.info(f"{'-'*70}")
                    
                    db_config = source_config[source_config['db_name'] == db].iloc[0]
                    dbtype = db_config['db_connection_type'].lower()
                    db_code = db_config['source_db_code']
                    
                    # Establish source connection
                    if dbtype == 'mssql':
                        connection_source = initialize_database_connection(
                            "mssql",
                            mssql_server=multi_db_source_host,
                            mssql_user=multi_db_source_user,
                            mssql_pass=multi_db_source_password,
                            mssql_db=db,
                            mssql_port=multi_db_source_port
                        )
                    elif dbtype == 'postgres':
                        connection_source = initialize_database_connection(
                            "postgres",
                            pg_dbname=db,
                            pg_user=multi_db_source_user,
                            pg_password=multi_db_source_password,
                            pg_host=multi_db_source_host,
                            pg_port=multi_db_source_port
                        )
                    elif dbtype == 'mysql':
                        connection_source = initialize_database_connection(
                            "mysql",
                            host=multi_db_source_host,
                            user=multi_db_source_user,
                            password=multi_db_source_password,
                            database=db,
                            port=multi_db_source_port
                        )
                    else:
                        raise Exception(f"Unsupported database type: {dbtype}")
                    
                    logging.info(f"Connected to source: {db}")
                    
                    # Create process for this DB
                    db_etl_name = f"{etl_name}_{source_system}_{db}"
                    db_process_id = update_audit_master_tbl(
                        connection_raw_db,
                        batch_run_id=batch_run_id,
                        etl_name=db_etl_name,
                        etl_start_time=etl_start_time,
                        etl_end_time=None,
                        status="In-Progress",
                        table_processed=0,
                        table_succeeded=0,
                        table_failed=0,
                        error_msg=None,
                        process_id=None,
                        etl_schedule_time=etl_start_time
                    )
                    logging.info(f"Process ID: {db_process_id}")
                    
                    # Filter and sort tables
                    db_etl_config = source_config[source_config['db_name'] == db]
                    db_etl_metainfo = db_etl_config.sort_values(
                        by=["load_sequence", "hierarchy_number"]
                    ).reset_index(drop=True)
                    
                    db_total_tables = 0
                    db_successful_tables = 0
                    db_failed_tables = 0
                    
                    # Process each table
                    for table_index, row in db_etl_metainfo.iterrows():
                        db_total_tables += 1
                        source_total_tables += 1
                        total_tables += 1
                        
                        table_process_start_time = datetime.now()
                        source_schema = row["source_schema_name"]
                        source_table = row["source_table_name"]
                        destination_table = row["target_table_name"]
                        operation = row["load_type"]
                        
                        logging.info(f"\nTable [{table_index+1}/{len(db_etl_metainfo)}]: {source_schema}.{source_table}")
                        
                        try:
                            # Upsert table audit
                            upsert_etl_audit_record(
                                connection_raw_db,
                                db_process_id,
                                batch_run_id,
                                source_system,
                                source_table,
                                destination_table,
                                operation,
                                db_code=db_code,
                                time_from=table_process_start_time,
                                input_row=0,
                                status="In-Progress",
                                time_to=None,
                                output_row=0,
                                error_code=None,
                                result=None,
                                etl_end_time=None,
                                etl_start_time=etl_start_time
                            )
                            
                            # Get last processed date for DELTA
                            last_processed_date_val = None
                            if run_type.upper() == "DELTA" and operation.lower() in ("incremental", "append"):
                                last_processed_date_val = get_last_processed_date(
                                    connection_raw_db, 
                                    source_table, 
                                    db_code,
                                    upper_bound_time
                                )
                                last_processed_date_val = '2026-01-01 00:00:00.000' # --------------------------Debugging                                
                            elif run_type.upper() == "DELTA" and operation.lower() == "full":
                                logging.info("DELTA run with FULL operation: processing all data up to upper_bound_time")
                                truncate_query = f"""
                                    TRUNCATE TABLE "{row['target_schema_name']}"."{destination_table}" RESTART IDENTITY;
                                    """
                                with connection_raw_db.cursor() as cursor:
                                    cursor.execute(truncate_query)
                                    connection_raw_db.commit()
                                    logging.info(f"TRUNCATED: {row['target_schema_name']}.{destination_table} (DELTA run with FULL operation)")
                        
                            
                            batch_size = int(row['batch_size']) if pd.notnull(row['batch_size']) else 100000
                            
                            # Determine if first DB
                            table_key = f"{row['target_schema_name']}.{destination_table}"
                            is_first_db_for_multi = table_key not in multi_db_truncation_status
                            
                            if is_first_db_for_multi:
                                multi_db_truncation_status[table_key] = {
                                    'truncated': True,
                                    'databases_processed': [db]
                                }
                            else:
                                multi_db_truncation_status[table_key]['databases_processed'].append(db)
                            
                            # Process table
                            total_fetched, total_processed, final_status, error_message = process_table(
                                connection_source_db=connection_source,
                                connection_raw_db=connection_raw_db,
                                row=row,
                                batch_size=batch_size,
                                process_id=db_process_id,
                                batch_run_id=batch_run_id,
                                run_type=run_type,
                                last_processed_date_val=last_processed_date_val,
                                upper_bound_time=upper_bound_time,
                                dbtype=dbtype,
                                is_first_db_for_multi=is_first_db_for_multi
                            )
                            
                            table_end_time = datetime.now()
                            
                            if final_status.lower() in ['success', 'partial success']:
                                db_successful_tables += 1
                                source_successful_tables += 1
                                successful_tables += 1
                                result_status = "Successful"
                                logging.info(f"SUCCESS: {source_table}")
                            else:
                                db_failed_tables += 1
                                source_failed_tables += 1
                                failed_tables += 1
                                result_status = "Unsuccessful"
                                logging.error(f"FAILED: {source_table}")
                            
                            # Update audit
                            upsert_etl_audit_record(
                                connection_raw_db,
                                db_process_id,
                                batch_run_id,
                                source_system,
                                source_table,
                                destination_table,
                                operation,
                                db_code=db_code,
                                time_from=None,
                                input_row=total_fetched,
                                status=final_status,
                                time_to=table_end_time,
                                output_row=total_processed,
                                error_code=error_message,
                                result=result_status,
                            )
                            
                        except Exception as table_error:
                            db_failed_tables += 1
                            source_failed_tables += 1
                            failed_tables += 1
                            table_error_message = str(table_error)
                            table_end_time = datetime.now()
                            
                            logging.error(f"ERROR: {source_table} | {table_error_message}")
                            
                            try:
                                upsert_etl_audit_record(
                                    connection_raw_db,
                                    db_process_id,
                                    batch_run_id,
                                    source_system,
                                    source_table,
                                    destination_table,
                                    operation,
                                    db_code=db_code,
                                    time_from=None,
                                    input_row=0,
                                    status="Failed",
                                    time_to=table_end_time,
                                    output_row=0,
                                    error_code=table_error_message,
                                    result="Unsuccessful",
                                )
                            except:
                                pass
                    
                    # Close DB connection
                    connection_source.close()
                    logging.info(f"Closed connection: {db}")
                    
                    # Update master audit for this DB
                    db_etl_end_time = datetime.now()
                    
                    if db_failed_tables == 0:
                        db_status = "Success"
                        db_error_msg = None
                    elif db_successful_tables > 0:
                        db_status = "Partial Success"
                        db_error_msg = f"{db_failed_tables}/{db_total_tables} tables failed"
                    else:
                        db_status = "Failed"
                        db_error_msg = "All tables failed"
                    
                    update_audit_master_tbl(
                        connection_raw_db,
                        batch_run_id=batch_run_id,
                        etl_name=db_etl_name,
                        etl_start_time=None,
                        etl_end_time=db_etl_end_time,
                        status=db_status,
                        table_processed=db_total_tables,
                        table_succeeded=db_successful_tables,
                        table_failed=db_failed_tables,
                        error_msg=db_error_msg,
                        process_id=db_process_id,
                    )
                
                # Multi-DB source result
                source_results.append({
                    'source_system': source_system,
                    'is_multi_db': True,
                    'databases': db_list,
                    'status': 'Success' if source_failed_tables == 0 else 'Partial Success' if source_successful_tables > 0 else 'Failed',
                    'tables_processed': source_total_tables,
                    'tables_successful': source_successful_tables,
                    'tables_failed': source_failed_tables
                })
        
        # Overall completion
        etl_end_time = datetime.now()
        etl_duration = etl_end_time - etl_start_time
        
        if failed_tables == 0:
            overall_status = "Success"
            overall_error_message = None
        elif successful_tables > 0:
            overall_status = "Partial Success"
            overall_error_message = f"{failed_tables}/{total_tables} tables failed"
        else:
            overall_status = "Failed"
            overall_error_message = "All tables failed"
        
        logging.info(f"\n\n{'='*80}")
        logging.info(f"ETL COMPLETED: {etl_name}")
        logging.info(f"Run Type: {run_type}")
        logging.info(f"Status: {overall_status}")
        logging.info(f"Tables: {total_tables} | Success: {successful_tables} | Failed: {failed_tables}")
        logging.info(f"Duration: {etl_duration}")
        logging.info(f"{'='*80}")
        
        return {
            'status': overall_status,
            'run_type': run_type,
            'total_tables': total_tables,
            'successful_tables': successful_tables,
            'failed_tables': failed_tables,
            'duration': str(etl_duration),
            'batch_run_id': batch_run_id,
            'message': overall_error_message,
            'source_results': source_results
        }
        
    except Exception as main_error:
        etl_end_time = datetime.now()
        main_error_message = str(main_error)
        
        logging.error("=" * 80)
        logging.error(f"CRITICAL ETL FAILURE: {etl_name}")
        logging.error(f"Error: {main_error_message}")
        logging.error(f"Traceback: {traceback.format_exc()}")
        logging.error("=" * 80)
        
        return {
            'status': 'Failed',
            'total_tables': total_tables,
            'successful_tables': successful_tables,
            'failed_tables': failed_tables,
            'error': main_error_message
        }
    finally:
        try:
            if connection_raw_db:
                connection_raw_db.close()
                logging.info("Datalake connection closed")
        except Exception as e:
            logging.error(f"Error closing connection: {str(e)}")
        
        logging.info("=" * 80)
        logging.info(f"ETL FINISHED: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logging.info("=" * 80)


# ============================================================================
# ENTRY POINT FUNCTION
# ============================================================================

def execute_etl(
    pg_dl_dbname,
    pg_dl_user,
    pg_dl_password,
    pg_dl_host,
    pg_dl_port,
    sinlge_db_source_host,
    sinlge_db_source_port,
    sinlge_db_source_user,
    sinlge_db_source_password,
    source_name,
    table_type,
    multi_db_source_host=None,
    multi_db_source_port=None,
    multi_db_source_user=None,
    multi_db_source_password=None,
    source_db_code=None,
    table_name=None,
    etl_name="BRONZE_ETL",
):
    """
    Entry point function to execute ETL pipeline.
    Automatically detects FULL or DELTA load from global_batch_audit.
    
    Parameters:
        pg_dl_dbname (str): Datalake database name
        pg_dl_user (str): Datalake username
        pg_dl_password (str): Datalake password
        pg_dl_host (str): Datalake host
        pg_dl_port (str): Datalake port
        source_host (str): Source host
        source_port (str): Source port
        source_user (str): Source username
        source_password (str): Source password
        source_name (str/tuple): Source name(s) e.g. 'ExistingMB' or ('ExistingMB', 'Travea')
        table_type (str): 'Master' or 'Transaction'
        source_db_code (str/tuple, optional): DB code filter
        table_name (str/tuple, optional): Table name filter
        etl_name (str): ETL process name
    
    Returns:
        dict: Execution results
    """
    connection_details = {
        "pg_dl_connection": {
            "pg_dbname": pg_dl_dbname,
            "pg_user": pg_dl_user,
            "pg_password": pg_dl_password,
            "pg_host": pg_dl_host,
            "pg_port": pg_dl_port,
        },
    }
    
    return main(
        dl_connection_details=connection_details,
        sinlge_db_source_host=sinlge_db_source_host,
        sinlge_db_source_port=sinlge_db_source_port,
        sinlge_db_source_user=sinlge_db_source_user,
        sinlge_db_source_password=sinlge_db_source_password,
        source_name=source_name,
        table_type=table_type,
        table_name=table_name,
        source_db_code=source_db_code,
        etl_name=etl_name,
        multi_db_source_host=multi_db_source_host,
        multi_db_source_port=multi_db_source_port,
        multi_db_source_user=multi_db_source_user,
        multi_db_source_password=multi_db_source_password,        
    )

In [ ]:
# Data Lake Credentials
pg_dl_dbname = 'db_name'
pg_dl_user = 'user'
pg_dl_password = 'password'
pg_dl_host = 'host'
pg_dl_port = 'port'


# Source System Credentials
# MB Production Credentials
mssql_mb_server = 'server'
mssql_mb_user = 'user'
mssql_mb_password = 'password'
mssql_mb_port = 'port'

# Travea Production Credentials
pg_user='user'
pg_password='password'
pg_host='host'
pg_port='port'

# Configs:
source_name=('Travea',)
etl_name = 'BRONZE_ETL_DAILY_MASTER_RUN'
table_type = 'Master'


In [13]:
tables = (
    "tbl_ac_account",
    "tbl_ac_chart_of_ac",
    "tbl_ac_customer_data",
    "tbl_ac_mgmt_pandl_glcode",
    "tbl_ac_transaction",
    "tbl_ba_documents",
    "tbl_service",
    "tbl_service_fare",
    "tbl_ta_counter_staff",
    "tbl_ta_service",
    "tbl_ta_supp_doc_no"
)

In [14]:
result = execute_etl(
    pg_dl_dbname,
    pg_dl_user,
    pg_dl_password,
    pg_dl_host,
    pg_dl_port,
    sinlge_db_source_host=mssql_mb_server,
    sinlge_db_source_port=mssql_mb_port,
    sinlge_db_source_user=mssql_mb_user,
    sinlge_db_source_password=mssql_mb_password,
    source_name=source_name,
    table_type=table_type,
    multi_db_source_host=pg_host,
    multi_db_source_port=pg_port,
    multi_db_source_user=pg_user,
    multi_db_source_password=pg_password,
    source_db_code=('UTTL-UAE',),
    table_name=tables,
    etl_name=etl_name,
)

2026-03-06 09:26:46,212 - INFO - ================================================================================
2026-03-06 09:26:46,212 - INFO - ETL STARTED: BRONZE_ETL_DAILY_MASTER_RUN
2026-03-06 09:26:46,213 - INFO - Start Time: 2026-03-06 09:26:46
2026-03-06 09:26:46,213 - INFO - ================================================================================
2026-03-06 09:26:46,214 - INFO - Step 1: Establishing datalake connection...
2026-03-06 09:26:46,226 - INFO - PostgreSQL connection established successfully.
2026-03-06 09:26:46,226 - INFO - Datalake connection established
2026-03-06 09:26:46,227 - INFO - 
Step 2: Reading global batch audit...
2026-03-06 09:26:46,246 - INFO - Running batch found with batch_run_id: 26
2026-03-06 09:26:46,247 - INFO - Batch Run ID: 26
2026-03-06 09:26:46,248 - INFO - Run Type: FULL
2026-03-06 09:26:46,248 - INFO - Upper Bound Time: 2026-03-06 08:11:21.812934
2026-03-06 09:26:46,248 - INFO - Run Frequency: DAILY
2026-03-06 09:26:46,249 - INFO - 